# FireCrawl

## Websearch

In [ ]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

FIRECRAWL_API = os.environ["FIRECRAWL_API_KEY"]

url = "https://api.firecrawl.dev/v2/search"

payload = {
  "query": "what are the latest news about Jensen Huang?",
  "sources": [
    "web"
  ],
  "categories": [],
  "limit": 3,
  "scrapeOptions": {
    "onlyMainContent": True,
    "maxAge": 172800000,
    "parsers": [
      "pdf"
    ],
    "formats": []
  }
}

headers = {
    "Authorization": f"Bearer {FIRECRAWL_API}",
    "Content-Type": "application/json"
}

response = requests.post(url, json=payload, headers=headers)

print(response.json())


In [ ]:
response.json().keys()

In [ ]:
response.json()["data"].keys()

In [ ]:
search_results = (response.json()["data"]["web"])
len(search_results)

In [ ]:
search_results[0].keys()

In [ ]:
search_results[0]["url"]

In [ ]:
search_results[0]["title"]

In [ ]:
search_results[0]["position"]

In [ ]:
import json

with open("jensen_huang.json", "w", encoding="utf-8") as f:
    json.dump(search_results, f, ensure_ascii=False, indent=4)

In [ ]:
import json
with open("jensen_huang.json", encoding="utf-8") as f:
    data = json.load(f)

data    

## Webscraping

- scrape specific page

In [ ]:
from firecrawl import Firecrawl

app = Firecrawl()

data = app.scrape(
    "https://pydantic.dev/docs/ai/evals/evals/",
    only_main_content=True,
    max_age=172800000,
    parsers=["pdf"]
)

print(data)

In [ ]:
data.__dict__.keys()

In [ ]:
metadata = data.metadata
metadata.__dict__.keys()

In [ ]:
print(f"{metadata.title = }")
print(f"{metadata.description = }")
print(f"{metadata.url = }")
print(f"{metadata.language = }")
print(f"{metadata.keywords = }")
print(f"{metadata.robots = }")

In [ ]:
with open(f"{metadata.title.replace("|", "")}.md", "w", encoding="utf-8") as f:
    f.write(data.markdown)

## Crawling

In [ ]:
from firecrawl import Firecrawl
from firecrawl.v2.types import ScrapeOptions

app = Firecrawl()

scrape_opts = ScrapeOptions(
    only_main_content=True,
    max_age=172800000,
    parsers=["pdf"]
)

crawl_result = app.crawl(
    "https://www.myh.se/validering-och-seqf",
    sitemap="include",
    crawl_entire_domain=False,
    limit=10,
    scrape_options=scrape_opts
)


In [ ]:
crawl_result.__dict__.keys()

In [ ]:
crawl_result.credits_used

In [ ]:
crawl_result.total

In [61]:
from pathlib import Path

Path("myh").mkdir(exist_ok=True)

for document in crawl_result.data:
    metadata = document.metadata
    with open(f"myh/{metadata.title.replace('|', '')}.md", "w", encoding="utf-8") as f:
        f.write(document.markdown)